In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Simular um modelo de Ising pulsado com a função TEM

import TutorialFeedback from '@site/src/components/TutorialFeedback';



O método de mitigação de erros por rede tensorial (TEM) da Algorithmiq é um algoritmo híbrido quântico-clássico projetado para realizar a mitigação de ruído inteiramente na etapa de pós-processamento clássico. Com o TEM, você pode calcular os valores esperados dos observáveis, mitigando os inevitáveis erros induzidos por ruído no hardware quântico com maior precisão e eficiência de custo, tornando-o uma opção muito atraente para pesquisadores quânticos e profissionais da indústria.

Este tutorial demonstra como o TEM pode obter resultados significativos para a dinâmica de um sistema quântico, que seria inacessível sem mitigação de erros e que requer substancialmente mais recursos quânticos se outros métodos de mitigação de erros como PEC e ZNE forem utilizados.

*Estimativa de uso: este notebook utiliza aproximadamente 10 minutos de QPU em dispositivos Heron r3. O tempo de execução pode variar substancialmente dependendo do dispositivo escolhido. As estimativas de uso por seção podem ser encontradas abaixo.*
## Executar experimentos de física de muitos corpos com mitigação de erros via função TEM
Este tutorial é baseado na seguinte referência: [L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9). Esta referência discute uma simulação real em hardware quântico de até 91 qubits. Neste tutorial, recriamos uma simulação semelhante em um circuito de menor tamanho.

O modelo de Ising pulsado corresponde ao modelo de Ising usual:

$$ \hat{H}_{\text{I}} = J \sum_{n=0}^{N-2} \hat{Z}_n \hat{Z}_{n+1} + h \sum_{n=0}^{N-1} \hat{Z}_n $$

ao qual é aplicado um pulso transversal:

$$ \hat{H}_{K} = b \sum_{n=0}^{N-1} \hat{X}_n $$

O objetivo é simular a dinâmica de um estado sob o Hamiltoniano de Ising pulsado transversal, cuja evolução temporal pode ser implementada por um unitário de Floquet $\hat{U}_{\text{KI}} = e^{-i \hat{H}_K} e^{-i \hat{H}_I} $. O estado inicial a evoluir é aquele no qual o primeiro qubit está no estado $|+\rangle$, enquanto os demais estão emparelhados e configurados no estado de Bell $(|00\rangle + |11\rangle)/\sqrt{2}$.

A quantidade que queremos observar é a função de correlação. O [artigo de referência](https://www.nature.com/articles/s41567-025-03144-9) discute como essa quantidade pode ser reescrita como um operador de Pauli $\hat{X}$ no $n^{ésimo}$ qubit.
Após um número de passos de tempo físicos $t$, calculamos o valor do operador de Pauli $\hat{X}_{n=t}$.
Dependendo dos parâmetros do sistema, o valor deste observável é igual a um valor que pode ser calculado exatamente, ou apenas simulado por métodos aproximados. Especificamente, para $|J|=|b|=\pi/4$, é igual a $[\cos(2h)]^t$, que é o valor que usaremos para comparar os resultados deste tutorial. Além disso, em um dado passo de tempo $t$, $\langle\hat{X}_{n\neq t}\rangle$ é zero. Para detalhes sobre como obter esses valores, e para comparação com resultados de simulação clássica aproximada fora desses parâmetros, consulte [L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9).

O TEM funciona primeiro caracterizando o ruído de cada camada única de gates de dois qubits no circuito, bem como caracterizando o erro de leitura. Em seguida, o circuito é executado na máquina quântica. Por fim, a mitigação de erros por rede tensorial é realizada nos recursos clássicos no IBM Cloud&reg; e o valor mitigado é retornado. Neste exemplo, o circuito tem duas camadas únicas para caracterizar.
## Configuração
Como pré-requisito, certifique-se de que as dependências necessárias estejam instaladas.

In [ ]:
%pip install numpy matplotlib qiskit qiskit-ibm-catalog qiskit-ibm-runtime pylatexenc qiskit_qasm3_import

In [1]:
import os
from matplotlib import pyplot as plt
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit.qasm3 import load

from qiskit_ibm_catalog import QiskitFunctionsCatalog

## Mitigação de erros com TEM
Fornecemos aqui um circuito que implementa o modelo de Ising pulsado descrito acima. O circuito é preparado da seguinte forma. Primeiro, há uma fase de preparação de estado, na qual o primeiro qubit está no estado $|+\rangle$, enquanto os demais estão em pares de Bell $(|00\rangle + |11\rangle)/\sqrt{2}$. Isso é seguido pela estrutura em tijolos que implementa a evolução unitária $\hat{U}_{\text{KI}}$. O número de passos de tempo físicos corresponde a $t/2$ camadas de circuito.
O código a seguir baixa os dois arquivos QASM necessários para este tutorial.

In [ ]:
# Download required QASM files
import urllib

urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/swy5jtq309b0xpzluzlmsmj908yphes8.qasm",
    "ki_30q.qasm",
)
urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/et3gkodonw6gsp2trs43lzaozrdtiu7s.qasm",
    "ki_12q.qasm",
)

Podemos visualizar uma versão pequena do circuito, com 12 qubits e seis passos de tempo:

In [3]:
# Parameters of the kicked Ising model
h = 0.0
num_qubits = 12
t_steps = 6

# Load the circuit for the kicked Ising model
small_circuit = load("ki_12q.qasm")

# Draw the circuit
small_circuit.draw("mpl", scale=0.25, fold=-1)

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif)

Em seguida, construa o observável $\hat{X}_{n=t}$. Ele é construído como uma simples cadeia de Pauli com a ordem correspondente à usada pelo Qiskit:

In [3]:
def xt_observable(n_qubits, t_steps):
    pauli_str = "".join(["I" * t_steps, "X", "I" * (n_qubits - t_steps - 1)])
    pauli_str = pauli_str[::-1]  # Reverse the string to match qiskit order
    return SparsePauliOp(data=pauli_str, coeffs=1.0)

Em nosso pequeno exemplo de 12 qubits, o observável se parece com isto:

In [4]:
# Build the observable for the kicked Ising model
small_observable = xt_observable(n_qubits=12, t_steps=6)
print(small_observable)

SparsePauliOp(['IIIIIXIIIIII'],
              coeffs=[1.+0.j])


Qiskit Functions use PUBs as the way to collect the inputs. In our case, let's consider a single circuit and observable as our PUB:

In [ ]:
# Collect the input PUBs, in this case composed of a
# single circuit and observable
pubs = [(small_circuit, [small_observable])]

As funções Qiskit usam PUBs como forma de coletar as entradas. No nosso caso, vamos considerar um único circuito e observável como nosso PUB:

In [6]:
# Set IBM Quantum credentials and backend configuration
personal_token = os.environ.get(
    "QISKIT_IBM_TOKEN", "<API-KEY>"
)  # Replace with your personal token or set the environment variable
channel = "ibm_quantum_platform"
crn = "your_crn"  # Replace with the Cloud Resource Name (CRN)

# Select the QPU backend
backend_name = "ibm_qpu_name"  # Replace with your desired backend's name

Em seguida, obtemos acesso à função TEM. Primeiro configuramos a autenticação necessária para o IBM Cloud e selecionamos um backend entre os dispositivos disponíveis. O token, os backends disponíveis e os nomes de recursos de nuvem correspondentes (CRN) podem ser obtidos fazendo login em sua conta no [painel do IBM Quantum Platform](https://quantum.cloud.ibm.com/).

In [8]:
# Load the TEM function from the Qiskit Functions Catalog
catalog = QiskitFunctionsCatalog(
    channel=channel,
    token=personal_token,
    instance=crn,
)
tem = catalog.load("algorithmiq/tem")

Carregue a função TEM do [Qiskit Functions Catalog](https://quantum.cloud.ibm.com/functions):

In [9]:
tem_job = tem.run(pubs=pubs, backend_name=backend_name)

Agora podemos executar um experimento no circuito de Ising pulsado com mitigação de erros fornecida pelo TEM. Usando as configurações padrão, o TEM pode ser executado de forma simples com um tempo de execução QPU esperado de cerca de 2,5 minutos, dependendo do QPU:

In [10]:
print(tem_job.status())

QUEUED


Com as opções padrão, a função TEM executa três jobs no computador quântico: aprendizado de ruído, mitigação de leitura e amostragem do circuito. O número de shots usados por cada um deles pode ser alterado nas opções passadas para a função. Por padrão, esses parâmetros são configurados para atingir uma precisão de 0,05 nos valores esperados mitigados.
Você pode verificar o status do seu job no [painel do IBM Quantum Platform](https://quantum.cloud.ibm.com/) ou com:

In [11]:
# Get the results of the TEM job
tem_results = tem_job.result()[
    0
]  # Get the first and only result from the job
tem_evs = tem_results.data.evs[0]
tem_std = tem_results.data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 1.031 ± 0.046


We can also check how much quantum runtime was used for each call at [IBM Quantum Platform](https://quantum.cloud.ibm.com), or by inspecting the result metadata from the Python code.

In [12]:
# Get the TEM job runtime
tem_runtime = tem_job.result().metadata["resource_usage"][
    "RUNNING: EXECUTING_QPU"
]["QPU_TIME"]

print(f"TEM Runtime: {tem_runtime} seconds")

TEM Runtime: 155.0 seconds


Quando o status for `DONE`, podemos verificar os resultados brutos e mitigados. Os `tem_evs` definidos abaixo são os valores esperados dos observáveis solicitados, neste caso apenas um observável, $\langle \hat X_{n=t}\rangle$, e `tem_std` são os desvios padrão correspondentes.

In [ ]:
options = {
    "default_shots": 10_000,
    "tem_max_bond_dimension": 512,
    "tem_compression_cutoff": 1e-16,
    # This option helps optimizing the measurement
    # stage since the observable is strongly biased
    # toward the X operator for all the qubits.
    "compute_shadows_bias_from_observable": True,
    # set to True to keep experiment results private,
    # recommended for confidential circuits
    "private": False,
}

Custom options for the noise learner can also be passed. They follow the definitions used in the Qiskit Runtime [`NoiseLearnerOptions`](/docs/api/qiskit-ibm-runtime/options-noise-learner-options):

In [29]:
nl_options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32],
}

# add noise learning options to the overall options
options |= nl_options

Também podemos verificar quanto tempo de execução quântica foi usado para cada chamada no [IBM Quantum Platform](https://quantum.cloud.ibm.com), ou inspecionando os metadados de resultados do código Python.

In [30]:
tem_job_custom = tem.run(
    pubs=pubs, backend_name=backend_name, options=options
)

If the job is not set as private, we can recover the result at a later point. To do so, save the job ID printed here and use `tem_job_custom = catalog.get_job_by_id("your-job-id")`.

In [31]:
job_id = tem_job_custom.job_id
print(f"Job ID: {job_id}")

Job ID: 1ba10094-a541-457a-9287-dbd49306d12d


In [33]:
results_custom = tem_job_custom.result()
tem_evs = results_custom[0].data.evs[0]
tem_std = results_custom[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 0.956 ± 0.018


Opções personalizadas para o noise learner também podem ser passadas. Elas seguem as definições usadas no Qiskit Runtime [`NoiseLearnerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-noise-learner-options):

In [34]:
metadata_custom = results_custom[0].metadata

unmitigated_evs = metadata_custom["evs_non_mitigated"][0]
unmitigated_stds = metadata_custom["stds_non_mitigated"][0]
print(f"Unmitigated Result: {unmitigated_evs:.3f} ± {unmitigated_stds:.3f}")

# Exact result for the kicked Ising model from the reference paper
exact_evs = np.cos(2 * h) ** t_steps
print("Exact Result:", exact_evs)

Unmitigated Result: 0.894 ± 0.015
Exact Result: 1.0


In [35]:
# Plot comparing the different expectation values
plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/c3a2168d-98df-491e-a1f8-05de5684ab96-0.avif" alt="Output of the previous code cell" />

Se o job não estiver configurado como privado, podemos recuperar o resultado posteriormente. Para isso, salve o ID do job impresso aqui e use `tem_job_custom = catalog.get_job_by_id("your-job-id")`.

In [27]:
# Get the metadata of the TEM job
job_metadata = results_custom.metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

QPU Runtime: 342.0 seconds
Classical Runtime: 107.632604 seconds


## Scale TEM to large circuits

Large circuits can, in principle, be run with the TEM function. However, it is important to be aware of the of the limitations of the classical resources, as TEM is executed on IBM Cloud runners with potentially very long run times. For extremely large circuits, contact the TEM support team at [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi).

Here we run an example with a larger, utility-scale-sized 30-qubit circuit, optimizing the TEM parameters for speed rather than accuracy.

In [ ]:
# Kicked Ising model parameters
n_qubits = 30
t_steps = 15
h = 0.0

# Load the circuit for the kicked Ising model
circuit = load("ki_30q.qasm")


# Build the observable for the kicked Ising model
observable = xt_observable(n_qubits=n_qubits, t_steps=t_steps)

# Collect the input PUBs, in this case composed of a
# single circuit and observable
pubs = [(circuit, [observable])]

Let's define some performance-oriented options:

In [49]:
options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32, 64],
    "default_shots": 5_000,
    "tem_max_bond_dimension": 128,
    "tem_compression_cutoff": 1e-10,
    "compute_shadows_bias_from_observable": True,
    "private": False,
}

Finally, run the experiment, get the result, and visualize it. This will take approximately 3.5 QPU minutes.

In [50]:
tem_job_large = tem.run(pubs=pubs, backend_name=backend_name, options=options)

In [51]:
job_id = tem_job_large.job_id
print(f"Job ID: {job_id}")

Job ID: 9f3f190f-f4b0-4dcb-bb83-5f71f37d0d77


In [53]:
results_large = tem_job_large.result()
tem_evs = results_large[0].data.evs[0]
tem_std = results_large[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")


# Get the metadata of the TEM job
job_metadata = tem_job_large.result().metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

TEM Result: 0.794 ± 0.026
QPU Runtime: 203.0 seconds
Classical Runtime: 251.71805499999996 seconds


In [54]:
# Plot comparing the different expectation values
metadata_large = results_large[0].metadata
unmitigated_evs = metadata_large["evs_non_mitigated"][0]
unmitigated_stds = metadata_large["stds_non_mitigated"][0]

exact_evs = np.cos(2 * h) ** t_steps

plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/24894c44-e399-4b9d-a3ff-38a28ff32ece-0.avif" alt="Output of the previous code cell" />